In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("PySpark Advanced Joins Demo") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Spark Session Created Successfully!")

In [ ]:
employee_data = [
    (101, "Amit",   10, 50000),
    (102, "Neha",   20, 65000),
    (103, "Rahul",  10, 55000),
    (104, "Priya",  30, 70000),
    (105, "Karan",  40, 45000),
    (106, "Sneha",  None, 60000)
]

employee_columns = [
    "emp_id",
    "emp_name",
    "dept_id",
    "salary"
]

employees_df = spark.createDataFrame(
    employee_data,
    employee_columns
)

employees_df.show()

In [ ]:
department_data = [
    (10, "IT"),
    (20, "HR"),
    (30, "Finance"),
    (50, "Marketing")
]

department_columns = [
    "dept_id",
    "dept_name"
]

departments_df = spark.createDataFrame(
    department_data,
    department_columns
)

departments_df.show()

In [ ]:
inner_df = employees_df.join(
    departments_df,
    on="dept_id",
    how="inner"
)

inner_df.show()

In [ ]:
left_df = employees_df.join(
    departments_df,
    on="dept_id",
    how="left"
)

left_df.show()

In [ ]:
invalid_department_df = (
    employees_df
    .join(
        departments_df,
        on="dept_id",
        how="left"
    )
    .filter(col("dept_name").isNull())
)

invalid_department_df.show()

In [ ]:
right_df = employees_df.join(
    departments_df,
    on="dept_id",
    how="right"
)

right_df.show()

In [ ]:
empty_departments_df = (
    employees_df
    .join(
        departments_df,
        on="dept_id",
        how="right"
    )
    .filter(col("emp_id").isNull())
)

empty_departments_df.show()

In [ ]:
full_df = employees_df.join(
    departments_df,
    on="dept_id",
    how="full"
)

full_df.show()

In [ ]:
unmatched_df = (
    employees_df
    .join(
        departments_df,
        on="dept_id",
        how="full"
    )
    .filter(
        col("emp_id").isNull() |
        col("dept_name").isNull()
    )
)

unmatched_df.show()

In [ ]:
cross_df = employees_df.crossJoin(
    departments_df
)

cross_df.show()

In [ ]:
emp = employees_df.alias("emp")
dept = departments_df.alias("dept")
result_df = emp.join(
    dept,
    col("emp.dept_id") == col("dept.dept_id"),
    "inner"
)

result_df.show()

In [ ]:
result_df = (
    emp
    .join(
        dept,
        col("emp.dept_id") == col("dept.dept_id"),
        "inner"
    )
    .select(
        col("emp.emp_id"),
        col("emp.emp_name"),
        col("emp.dept_id"),
        col("dept.dept_name"),
        col("emp.salary")
    )
)

result_df.show()

In [ ]:
high_salary_df = (
    employees_df.alias("emp")
    .join(
        departments_df.alias("dept"),
        col("emp.dept_id") == col("dept.dept_id"),
        "inner"
    )
    .filter(
        col("emp.salary") > 55000
    )
    .select(
        col("emp.emp_id"),
        col("emp.emp_name"),
        col("dept.dept_name"),
        col("emp.salary")
    )
)

high_salary_df.show()

In [ ]:
from pyspark.sql.functions import count, avg, max, min

department_summary_df = (
    employees_df.alias("emp")
    .join(
        departments_df.alias("dept"),
        col("emp.dept_id") == col("dept.dept_id"),
        "inner"
    )
    .groupBy(
        col("dept.dept_name")
    )
    .agg(
        count("emp.emp_id").alias("employee_count"),
        avg("emp.salary").alias("average_salary"),
        min("emp.salary").alias("minimum_salary"),
        max("emp.salary").alias("maximum_salary")
    )
)

department_summary_df.show()

In [11]:
employees_df.join(
    departments_df,
    employees_df.dept_id == departments_df.dept_id,
    "inner"
).show()

employees_df.join(
    departments_df,
    employees_df.dept_id == departments_df.dept_id,
    "left_semi"
).show()

employees_df.join(
    departments_df,
    employees_df.dept_id == departments_df.dept_id,
    "left_anti"
).show()


+------+--------+-------+------+-------+---------+
|emp_id|emp_name|dept_id|salary|dept_id|dept_name|
+------+--------+-------+------+-------+---------+
|   101|    Amit|     10| 50000|     10|       IT|
|   103|   Rahul|     10| 55000|     10|       IT|
|   102|    Neha|     20| 65000|     20|       HR|
|   104|   Priya|     30| 70000|     30|  Finance|
+------+--------+-------+------+-------+---------+

+------+--------+-------+------+
|emp_id|emp_name|dept_id|salary|
+------+--------+-------+------+
|   101|    Amit|     10| 50000|
|   103|   Rahul|     10| 55000|
|   102|    Neha|     20| 65000|
|   104|   Priya|     30| 70000|
+------+--------+-------+------+

+------+--------+-------+------+
|emp_id|emp_name|dept_id|salary|
+------+--------+-------+------+
|   105|   Karan|     40| 45000|
|   106|   Sneha|   NULL| 60000|
+------+--------+-------+------+

